# Train a Regression Model

In this notebook we will train a Linear Regression Model on the Green Taxi dataset. We will only use **one month** for the training and keep only a **small number of features**. 

We want the model to **predict the duration of a trip**. This can be useful for the taxi drivers to plan their trips, for the customers to know how long a trip will take but also for the taxi companies to plan their fleet. The first two predictions would need **real time predictions** because the duration of a trip is not known in advance. The last one could be done in **batch mode**, as it is more an analytical task that doesn't need to be done in real time.

Additionally, we will use MLflow to track the model training and log the model artifacts.

In [ ]:
import os
from dotenv import load_dotenv

import pandas as pd

import mlflow
from mlflow import MlflowClient

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [ ]:
year = 2024
month = 1
color = "green"

In [ ]:
# Download the data
if not os.path.exists(f"./data/{color}_tripdata_{year}-{month:02d}.parquet"):
    os.system(f"wget -P ./data https://d37ci6vzurychx.cloudfront.net/trip-data/{color}_tripdata_{year}-{month:02d}.parquet")

In [ ]:
# Load the data
df = pd.read_parquet(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")

In [ ]:
df.shape

Now we will set up the connection to MLflow. For that we have to create a `.env` file with the URI to the MLflow Server in GCP (this will be `http://<external-ip>:5000`). You can simply run:

```bash
echo "MLFLOW_TRACKING_URI=http://<external-ip>:5000" > .env
```

We will create a new experiment called `green-taxi-monitoring` in MLflow to track our model training runs:
+ **developer** tag should be your name
+ **artifact_location** tag should point to your GCS bucket, supposed that you have created  a folder called `models` in it, then it should be `gs://<GCS bucket>/models`

In [ ]:
# Load environment variables from .env file
load_dotenv()
# Get the MLflow tracking URI from environment variables
MLFLOW_TRACKING_URI=os.getenv("MLFLOW_TRACKING_URI")

# Set up the connection to MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Give the experiment a name
experiment_name = f"green-taxi-monitoring"
# Get the experiment by name
exp = mlflow.get_experiment_by_name(experiment_name)

# Create the experiment if it does not exist
if exp is None:
    mlflow.create_experiment(name=experiment_name, 
                         tags={"developer": "<your-name>"},
                         artifact_location="gs://<GCS bucket>/models"
                         )
else:
    experiment_id = exp.experiment_id
    print(f"Using existing experiment: {experiment_name} (ID: {experiment_id})")

# Set the experiment for logging
mlflow.set_experiment(experiment_name)

If everything went well, you should be able to see the experiment now in the MLflow UI at `http://<external-ip>:5000`.

Let's start now by looking at the data:

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Look for missing values
df.isnull().sum()

Nearly all features seem to be in the correct type and we have only missings in features that we will not use for the model training. For predicting the duration of a trip, we will use the following features:

- `PULocationID`: The pickup location ID
- `DOLocationID`: The dropoff location ID
- `trip_distance`: The distance of the trip in miles
- `passenger_count`: The number of passengers
- `fare_amount`: The fare amount in USD
- `total_amount`: The total amount in USD

But first we have to calculate the duration of the trip in minutes because it is our target. For that we will use the `tpep_pickup_datetime` and `tpep_dropoff_datetime` columns. We will also remove all trips that have a duration of 0 and that are longer than 1 hours to remove outliers.

In [ ]:
features = ["PULocationID", "DOLocationID", "trip_distance", "passenger_count", "fare_amount", "total_amount"]
target = 'duration'

In [ ]:
# Calculate the trip duration in minutes and drop trips that are less than 1 minute and more than 2 hours
def calculate_trip_duration_in_minutes(df):
    df["duration"] = (df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]).dt.total_seconds() / 60
    df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]
    df = df[(df['passenger_count'] > 0) & (df['passenger_count'] < 8)]
    df = df[features + [target]]
    return df

In [ ]:
df_processed = calculate_trip_duration_in_minutes(df)

In [ ]:
df_processed.head()

In [ ]:
df_processed.duration.describe()

In [ ]:
df_processed.duration.hist()

Now that we have the dataframe that we want to train our model on, we need to split it into a train and test set. We will use 80% of the data for training and 20% for testing.

In [ ]:
y = df_processed["duration"]
X = df_processed.drop(columns=["duration"])

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

And now we can train the model and track the experiment with MLflow. We will set tags to the experiment to make it easier to find later.

- `model`: `linear regression`
- `dataset`: `green-taxi`
- `developer`: `your name`
- `train_size`: The size of the train set
- `test_size`: The size of the test set
- `features`: The features that we used for training
- `target`: The target that we want to predict
- `year`: The year of the data
- `month`: The month of the data

We could also log the model parameters but Linear Regression doesn't have any.

And finally we will log the metrics:

- `rmse`: The root mean squared error

We will also log the model artifacts. For that we will need to set the **service account JSON** that we downloaded before as the environment variable `GOOGLE_APPLICATION_CREDENTIALS`.

Add in the `.env` file the following line:

```bash
GOOGLE_APPLICATION_CREDENTIALS=/path/to/your/service-account-file.json
```
In case yyou have stored the file in the `sa_key` folder in the repo you can add:

```bash
GOOGLE_APPLICATION_CREDENTIALS=sa_key/your-service-account-file.json
```

In [ ]:
gcp_key = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = gcp_key

In [ ]:
with mlflow.start_run():
    
    tags = {
        "model": "linear regression",
        "developer": "<your-name>",
        "dataset": f"{color}-taxi",
        "year": year,
        "month": month,
        "features": features,
        "target": target
    }
    mlflow.set_tags(tags)
    
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    
    y_pred = lr.predict(X_test)
    rmse = root_mean_squared_error(y_test, y_pred)
    mlflow.log_metric("rmse", rmse)
    
    mlflow.sklearn.log_model(sk_model=lr, name="mlflow-model")
    run_id = mlflow.active_run().info.run_id

    
    # Register model name in the model registry
    client = MlflowClient()
    model_name = "green-taxi-ride-duration-1"
    tags = {"LinearRegression.framework": "Scikit-Learn"}
    desc = "Model to predict ride duration in NYC"
    client.create_registered_model(name=model_name,tags=tags,description=desc)

    # Create a new version of the rfr model under the registered model name
    model_uri = f"runs:/{run_id}/mlflow-model"
    mv = client.create_model_version(
        name=model_name,
        source=model_uri,
        run_id=run_id
    )

    
    # Set registered model alias
    new_alias = "production"
    client.set_registered_model_alias(
        name=model_name,
        version=mv.version,
        alias=new_alias
    )


You should now see your run in the MLflow UI. Under the created experiment, you can also see the logged tags, the metric and the saved model.

![](images/monitoring_mlflow_registered_model.png)

And you can see what you need to do to load the model in an API or script in the UI as long as the application has access to MLflow.